# Notebook 02: Competing Risks Model
## TransPlan Validation & Reproducibility Pack (Phase 4 M5)

This notebook documents the **competing risks model** that determines patient outcomes on the transplant waitlist. While Notebook 01 covered wait-time distributions in isolation, this notebook shows how three competing events interact:

1. **Transplant** — the desired outcome (log-normal wait time from Notebook 01)
2. **Mortality** — death while waiting (exponential with organ/urgency-adjusted rate)
3. **Delisting** — removal from waitlist for other reasons (exponential)

The event that occurs **first** determines the patient's outcome. This is the core of TransPlan's Monte Carlo simulation engine.

### Contents
1. Model specification and assumptions
2. Competing risks parameters (mortality & delisting rates)
3. Analytical P(transplant) vs. pure wait-time P(transplant)
4. Monte Carlo simulation walkthrough (single city)
5. Multi-horizon Brier score calibration (6/12/24/36 months)
6. Outcome breakdown visualization (stacked area)
7. City-level variation in competing risks impact

### Data Sources
- Mortality/delisting rates: SRTR PSR Table B7, January 2025 release
- Wait-time distributions: SRTR PSR Table B10 (see Notebook 01)

In [1]:
# --- Setup ---
import sys, os, json, hashlib
from pathlib import Path

REPO_ROOT = Path(os.getcwd()).parent if "notebooks" in os.getcwd() else Path(os.getcwd())
sys.path.insert(0, str(REPO_ROOT / "backend"))
os.chdir(REPO_ROOT / "backend")

import numpy as np
import pandas as pd
import scipy
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from scipy.integrate import quad

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 120

FIGURES_DIR = REPO_ROOT / "notebooks" / "figures"
FIGURES_DIR.mkdir(exist_ok=True)

RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)

# Log data hashes
for fname in ["wait-time-distributions.json", "competing-risks.json"]:
    fp = REPO_ROOT / "data" / fname
    h = hashlib.sha256(fp.read_bytes()).hexdigest()[:12]
    print(f"{fname}: SHA-256={h}")

print(f"\nPython: {sys.version.split()[0]}, NumPy: {np.__version__}, SciPy: {scipy.__version__}")
print(f"RNG seed: {RNG_SEED}")

wait-time-distributions.json: SHA-256=289de3ad599a
competing-risks.json: SHA-256=be30a1e9a785

Python: 3.14.3, NumPy: 2.4.2, SciPy: 1.17.1
RNG seed: 42


## 1. Model Specification

### Three Independent Competing Events

For each Monte Carlo iteration, three event times are drawn independently:

$$T_{\text{transplant}} \sim \text{LogNormal}(\mu, \sigma)$$
$$T_{\text{mortality}} \sim \text{Exponential}(\lambda_m)$$
$$T_{\text{delisting}} \sim \text{Exponential}(\lambda_d)$$

The outcome is determined by whichever event occurs first:

$$T_{\text{event}} = \min(T_{\text{transplant}}, T_{\text{mortality}}, T_{\text{delisting}})$$
$$\text{outcome} = \arg\min(T_{\text{transplant}}, T_{\text{mortality}}, T_{\text{delisting}})$$

### Key Assumption: Independence

The three events are modeled as **independent**. This is a simplification — in reality, a patient's declining health could simultaneously increase mortality risk AND reduce likelihood of transplant (surgeons may consider them too sick). This assumption is standard in SRTR methodology and is documented as a known limitation.

### Rate Adjustments

Mortality rates are adjusted by:
- **Urgency level** (1-4): higher urgency → higher mortality risk
- **MELD score** (liver only): higher MELD → higher mortality
- **City factor**: center-level quality adjustment

Delisting rates are adjusted by:
- **City factor** only (organ-level national rate × center adjustment)

In [2]:
# Load competing risks data and import services
from services.distributions import get_wait_time_distribution
from services.competing_risks import (
    get_annual_mortality_rate, get_annual_delisting_rate, get_organ_risks
)

with open(REPO_ROOT / "data" / "competing-risks.json") as f:
    risks_data = json.load(f)

organs = ["kidney", "liver", "heart", "lung", "pancreas", "intestine"]

# Display national rates
rates_table = []
for organ in organs:
    r = risks_data[organ]
    rates_table.append({
        "Organ": organ.title(),
        "Annual Mortality Rate": r["annual_mortality_rate"],
        "Annual Delisting Rate": r["annual_delisting_rate"],
        "Monthly Mortality": round(r["annual_mortality_rate"] / 12, 5),
        "Monthly Delisting": round(r["annual_delisting_rate"] / 12, 5),
        "Mean Time to Death (mo)": round(12 / r["annual_mortality_rate"], 1),
        "Mean Time to Delist (mo)": round(12 / r["annual_delisting_rate"], 1),
    })

rates_df = pd.DataFrame(rates_table)
print("=== National Competing Risks Parameters (SRTR Table B7, January 2025) ===\n")
print(rates_df.to_string(index=False))

=== National Competing Risks Parameters (SRTR Table B7, January 2025) ===

    Organ  Annual Mortality Rate  Annual Delisting Rate  Monthly Mortality  Monthly Delisting  Mean Time to Death (mo)  Mean Time to Delist (mo)
   Kidney                 0.0194                 0.0425            0.00162            0.00354                    618.6                     282.4
    Liver                 0.0447                 0.1034            0.00372            0.00862                    268.5                     116.1
    Heart                 0.0288                 0.0821            0.00240            0.00684                    416.7                     146.2
     Lung                 0.0259                 0.0588            0.00216            0.00490                    463.3                     204.1
 Pancreas                 0.0279                 0.1734            0.00233            0.01445                    430.1                      69.2
Intestine                 0.0296                 0.0741

## 2. Urgency Multipliers

Higher-urgency patients face greater mortality risk. The urgency multiplier is applied to the base annual mortality rate. Delisting rates are not adjusted by urgency.

In [3]:
# Urgency mortality multiplier heatmap
urgency_levels = ["1", "2", "3", "4"]
urg_matrix = []
for organ in organs:
    mults = risks_data[organ].get("urgency_mortality_multipliers", {})
    row = [mults.get(u, 1.0) for u in urgency_levels]
    urg_matrix.append(row)

urg_df = pd.DataFrame(urg_matrix, index=[o.title() for o in organs], 
                       columns=["Urgency 1\n(low)", "Urgency 2\n(standard)", "Urgency 3\n(high)", "Urgency 4\n(critical)"])

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(urg_df, annot=True, fmt=".1f", cmap="YlOrRd", center=1.0,
            linewidths=1, linecolor="white", ax=ax,
            cbar_kws={"label": "Mortality Rate Multiplier"})
ax.set_title("Urgency-Adjusted Mortality Multipliers by Organ\n(Applied to annual mortality rate; 1.0 = standard urgency)", 
             fontsize=13, fontweight="bold")
ax.set_ylabel("")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "02-urgency-multipliers.png", bbox_inches="tight", dpi=150)
plt.show()

/var/folders/2v/mppjqytd61b5lt_ymg09qff80000gn/T/ipykernel_59655/2019566101.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Impact of Competing Risks on Transplant Probability

Competing risks **reduce** the probability of transplant because some patients die or are delisted before an organ becomes available. This section compares:
- **Pure wait-time P(transplant ≤ t)**: from the log-normal CDF alone (Notebook 01)
- **Competing risks P(transplant first AND ≤ t)**: accounting for mortality and delisting

The gap between these curves represents patients "lost" to death or delisting.

In [4]:
# Analytical P(transplant first AND within t) via numerical integration
def p_transplant_analytical(organ, blood_type, city, t_max, urgency=2, 
                              cpra=None, meld=None, las=None, n_points=500):
    """Compute P(transplant first AND within t) for a range of t values."""
    dist = get_wait_time_distribution(organ, blood_type, city, cpra, meld, las)
    annual_mort = get_annual_mortality_rate(organ, city, urgency, meld)
    annual_delist = get_annual_delisting_rate(organ, city)
    monthly_mort = annual_mort / 12.0
    monthly_delist = annual_delist / 12.0
    
    t_vals = np.linspace(0.01, t_max, n_points)
    p_vals = np.zeros(n_points)
    
    for i, t in enumerate(t_vals):
        def integrand(tau):
            f_t = float(dist.pdf(tau))
            s_m = np.exp(-monthly_mort * tau)
            s_d = np.exp(-monthly_delist * tau)
            return f_t * s_m * s_d
        p_vals[i], _ = quad(integrand, 0, t, limit=100)
    
    return t_vals, np.clip(p_vals, 0, 1)

# Compare pure CDF vs competing risks for each organ (kidney O+ Chicago, urgency 2)
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
colors = sns.color_palette("deep", 6)

for idx, organ in enumerate(organs):
    ax = axes[idx // 3, idx % 3]
    cfg = {"kidney": {"cpra": 30}, "liver": {"meld": 20}, "lung": {"las": 50.0}}.get(organ, {})
    
    dist = get_wait_time_distribution(organ, "O+", "Chicago", **cfg)
    t = np.linspace(0.01, 60, 300)
    
    # Pure wait-time CDF
    pure_cdf = dist.cdf(t)
    
    # Competing risks P(transplant)
    t_cr, p_cr = p_transplant_analytical(organ, "O+", "Chicago", 60, **cfg)
    
    ax.fill_between(t, pure_cdf, alpha=0.15, color=colors[idx])
    ax.plot(t, pure_cdf, color=colors[idx], linewidth=2, linestyle="--", alpha=0.6, label="Pure wait-time")
    ax.plot(t_cr, p_cr, color=colors[idx], linewidth=2.5, label="With competing risks")
    
    # Shade the gap
    p_cr_interp = np.interp(t, t_cr, p_cr)
    ax.fill_between(t, p_cr_interp, pure_cdf, alpha=0.2, color="red", label="Lost to death/delisting")
    
    # Annotate 24-month gap
    p24_pure = dist.cdf(24)
    p24_cr = np.interp(24, t_cr, p_cr)
    gap = p24_pure - p24_cr
    ax.annotate(f"Gap at 24mo:\n{gap:.1%}", xy=(24, (p24_pure + p24_cr)/2),
                fontsize=8, ha="left", color="red", fontweight="bold")
    
    ax.set_title(f"{organ.title()}", fontsize=13, fontweight="bold")
    ax.set_xlabel("Months")
    ax.set_ylabel("P(transplant)")
    ax.set_xlim(0, 60)
    ax.set_ylim(0, 1)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
    if idx == 0:
        ax.legend(fontsize=8, loc="lower right")

fig.suptitle("Effect of Competing Risks on Transplant Probability\n"
             "(O+ blood, Chicago, standard urgency — dashed = pure wait-time, solid = with mortality/delisting)",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "02-competing-risks-impact.png", bbox_inches="tight", dpi=150)
plt.show()
print("Saved: figures/02-competing-risks-impact.png")

Saved: figures/02-competing-risks-impact.png


/var/folders/2v/mppjqytd61b5lt_ymg09qff80000gn/T/ipykernel_59655/785199237.py:70: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Stacked Outcome Breakdown Over Time

At any time horizon t, the four possible patient states must sum to 100%:
- P(transplanted by t)
- P(died by t)
- P(delisted by t)
- P(still waiting at t)

This stacked area chart shows how these probabilities evolve over 5 years for a kidney O+ patient.

In [5]:
# Monte Carlo stacked outcome simulation for kidney O+ Chicago
N_ITER = 10_000
organ, bt, city = "kidney", "O+", "Chicago"

dist = get_wait_time_distribution(organ, bt, city, cpra=30)
annual_mort = get_annual_mortality_rate(organ, city, urgency=2)
annual_delist = get_annual_delisting_rate(organ, city)

# Draw all three event times
transplant_times = dist.rvs(size=N_ITER, random_state=rng)
mortality_times = rng.exponential(scale=12.0 / annual_mort, size=N_ITER)
delisting_times = rng.exponential(scale=12.0 / annual_delist, size=N_ITER)

all_times = np.stack([transplant_times, mortality_times, delisting_times], axis=1)
event_times = np.min(all_times, axis=1)
outcomes = np.argmin(all_times, axis=1)  # 0=transplant, 1=death, 2=delisted

# Compute stacked probabilities at each time point
t_vals = np.linspace(0, 60, 200)
p_transplant = np.array([np.mean((outcomes == 0) & (event_times <= t)) for t in t_vals])
p_death = np.array([np.mean((outcomes == 1) & (event_times <= t)) for t in t_vals])
p_delist = np.array([np.mean((outcomes == 2) & (event_times <= t)) for t in t_vals])
p_waiting = 1.0 - p_transplant - p_death - p_delist

fig, ax = plt.subplots(figsize=(14, 7))
ax.stackplot(t_vals, p_transplant, p_death, p_delist, p_waiting,
             labels=["Transplanted", "Died on waitlist", "Delisted", "Still waiting"],
             colors=["#2ca02c", "#d62728", "#ff7f0e", "#7f7f7f"], alpha=0.8)

# Mark 24-month cross-section
ax.axvline(24, color="black", linestyle="--", linewidth=1.5, alpha=0.5)
p24_t = np.interp(24, t_vals, p_transplant)
p24_d = np.interp(24, t_vals, p_death)
p24_dl = np.interp(24, t_vals, p_delist)
p24_w = np.interp(24, t_vals, p_waiting)
ax.text(25, 0.95, f"At 24 months:\nTransplant: {p24_t:.1%}\nDeath: {p24_d:.1%}\n"
        f"Delisted: {p24_dl:.1%}\nWaiting: {p24_w:.1%}", fontsize=10,
        va="top", bbox=dict(boxstyle="round,pad=0.4", facecolor="white", alpha=0.9))

ax.set_xlabel("Months on Waitlist", fontsize=12)
ax.set_ylabel("Cumulative Probability", fontsize=12)
ax.set_title(f"Competing Risks Outcome Breakdown: {organ.title()} {bt}, {city}\n"
             f"(n={N_ITER:,} Monte Carlo iterations, cPRA=30, urgency=2)",
             fontsize=14, fontweight="bold")
ax.legend(loc="center right", fontsize=11, framealpha=0.9)
ax.set_xlim(0, 60)
ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))

plt.tight_layout()
fig.savefig(FIGURES_DIR / "02-stacked-outcomes.png", bbox_inches="tight", dpi=150)
plt.show()
print("Saved: figures/02-stacked-outcomes.png")
print(f"\nVerification: probabilities sum to {p24_t + p24_d + p24_dl + p24_w:.4f} at 24 months (should be ~1.0)")

Saved: figures/02-stacked-outcomes.png

Verification: probabilities sum to 1.0000 at 24 months (should be ~1.0)


/var/folders/2v/mppjqytd61b5lt_ymg09qff80000gn/T/ipykernel_59655/1228784622.py:52: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Multi-Horizon Brier Score Calibration

The Brier score measures calibration: does the Monte Carlo engine's predicted P(transplant) match the analytical expectation from the same parameters? We extend Phase 2 M7's single-horizon (12mo) Brier score to all four horizons (6/12/24/36 months).

$$\text{BS} = \frac{1}{N} \sum_{i=1}^{N} (P_{\text{MC},i} - P_{\text{analytical},i})^2$$

where $N$ = 22 cities, and BS < 0.001 indicates excellent self-consistency.

In [6]:
# Multi-horizon Brier score using the production Monte Carlo engine
from services.monte_carlo import simulate, CITIES
from models.schemas import PatientProfile

def compute_multi_horizon_brier(organ, blood_type, urgency=2, n_iter=2000, **kwargs):
    """Compute Brier scores at 6/12/24/36 month horizons."""
    patient = PatientProfile(
        organ=organ, blood_type=blood_type, age=45, sex="male",
        urgency=urgency, **kwargs
    )
    sim = simulate(patient, n_iterations=n_iter)
    
    horizons = [6, 12, 24, 36]
    results = {}
    
    for h in horizons:
        squared_errors = []
        for cp in sim.cities:
            # MC prediction
            p_mc = getattr(cp, f"p_transplant_{h}mo")
            
            # Analytical prediction
            dist = get_wait_time_distribution(organ, blood_type, cp.city, 
                                               kwargs.get("cpra"), kwargs.get("meld"), kwargs.get("las"))
            annual_mort = get_annual_mortality_rate(organ, cp.city, urgency, kwargs.get("meld"))
            annual_delist = get_annual_delisting_rate(organ, cp.city)
            m_rate = annual_mort / 12.0
            d_rate = annual_delist / 12.0
            
            def integrand(tau):
                return float(dist.pdf(tau)) * np.exp(-m_rate * tau) * np.exp(-d_rate * tau)
            
            p_an, _ = quad(integrand, 0, h, limit=100)
            p_an = float(np.clip(p_an, 0, 1))
            
            squared_errors.append((p_mc - p_an) ** 2)
        
        results[h] = np.mean(squared_errors)
    
    return results

# Run for all 6 organs
organ_configs = [
    {"organ": "kidney", "blood_type": "O+", "cpra": 30},
    {"organ": "liver", "blood_type": "A+", "meld": 20},
    {"organ": "heart", "blood_type": "O+"},
    {"organ": "lung", "blood_type": "B+", "las": 50.0},
    {"organ": "pancreas", "blood_type": "A+"},
    {"organ": "intestine", "blood_type": "A+"},
]

print("=== Multi-Horizon Brier Score Calibration (2000 iterations) ===\n")
brier_results = []
for cfg in organ_configs:
    organ = cfg.pop("organ")
    bt = cfg.pop("blood_type")
    scores = compute_multi_horizon_brier(organ, bt, **cfg)
    brier_results.append({"Organ": organ.title(), **{f"BS@{h}mo": round(s, 6) for h, s in scores.items()}})
    print(f"  {organ.title():10s}: " + ", ".join(f"{h}mo={s:.6f}" for h, s in scores.items()))

brier_df = pd.DataFrame(brier_results)
print(f"\n{brier_df.to_string(index=False)}")
print(f"\nAll Brier scores < 0.001: {all(v < 0.001 for r in brier_results for k, v in r.items() if k.startswith('BS'))}")

=== Multi-Horizon Brier Score Calibration (2000 iterations) ===



  Kidney    : 6mo=0.000015, 12mo=0.000065, 24mo=0.000146, 36mo=0.000199


  Liver     : 6mo=0.000102, 12mo=0.000068, 24mo=0.000042, 36mo=0.000032


  Heart     : 6mo=0.000111, 12mo=0.000057, 24mo=0.000042, 36mo=0.000046


  Lung      : 6mo=0.000023, 12mo=0.000009, 24mo=0.000011, 36mo=0.000010


  Pancreas  : 6mo=0.000015, 12mo=0.000045, 24mo=0.000080, 36mo=0.000087


  Intestine : 6mo=0.000073, 12mo=0.000227, 24mo=0.000136, 36mo=0.000114

    Organ   BS@6mo  BS@12mo  BS@24mo  BS@36mo
   Kidney 0.000015 0.000065 0.000146 0.000199
    Liver 0.000102 0.000068 0.000042 0.000032
    Heart 0.000111 0.000057 0.000042 0.000046
     Lung 0.000023 0.000009 0.000011 0.000010
 Pancreas 0.000015 0.000045 0.000080 0.000087
Intestine 0.000073 0.000227 0.000136 0.000114

All Brier scores < 0.001: True


## 6. Summary & Known Limitations

### Key Findings
- Competing risks reduce transplant probability by **2-8%** at 24 months for most organs
- **Kidney** patients experience the largest competing-risks impact due to long absolute wait times
- **Lung** patients experience minimal impact — LAS-driven short waits mean few patients die or are delisted before transplant
- Brier scores remain < 0.001 across all horizons and organs, confirming Monte Carlo engine self-consistency

### Limitations

| Limitation | Impact | Mitigation |
|-----------|--------|------------|
| Independence assumption | Overestimates transplant probability for sickest patients | Could add correlation via copula model |
| Constant hazard rates | Mortality/delisting rates don't change over time | Piecewise-constant or Weibull model could capture time-varying risk |
| No age/sex adjustment | Mortality rates are organ-level, not demographic-specific | Equity analysis (NB 06) uses demographic profiles |
| Single urgency level per simulation | Real patients may change urgency over time | Dynamic urgency updates would require more complex state model |

### Next Notebook
**Notebook 03: Cause-of-Death Donor Availability Model** — how regional cause-of-death patterns create geographic variation in organ supply, modifying wait times.